# Этап 2. Очистка данных Яндекс Недвижимости

**Цель:** исследовать объединённый датасет `real_estate_dataset`, выявить дубликаты, пропуски и выбросы, создать воспроизводимые функции очистки для дальнейшего запуска в Apache Airflow.

**Источник данных:** PostgreSQL, таблица `real_estate_dataset` (результат первого этапа ETL). Исходная таблица в процессе исследования **не изменяется**.

**Порядок работы:** загрузка данных → анализ и удаление дубликатов → проверка площадей и заполнение пропусков → анализ и фильтрация ценовых выбросов → итоговая проверка. Все выводы ниже относятся к проверенному исходному набору из 141 362 квартир.

## 1. Загрузка исходного датасета

### 1.1. Импорт библиотек и подключение к PostgreSQL

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path
import sys

# Работаем как при запуске Jupyter из корня проекта, так и из папки notebooks.
project_root = Path.cwd()
if not (project_root / "db.py").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

from db import engine

print("Библиотеки загружены")

Подключение к PostgreSQL успешно!
База данных: real_estate
Домов: 24620
Квартир: 141362
Библиотеки загружены


**Вывод.** Библиотеки для работы с данными загружены. Для подключения к PostgreSQL используется файл `db.py`. Пароль не записан в ноутбуке, а хранится отдельно в `.env`.

### 1.2. Загрузка данных

In [2]:
query = "SELECT * FROM real_estate_dataset"

df = pd.read_sql(query, engine)

print("Размер датасета:", df.shape)

display(df.head())

Размер датасета: (141362, 18)


,flat_id,building_id,floor,kitchen_area,living_area,rooms,is_apartment,studio,total_area,price,build_year,building_type_int,latitude,longitude,ceiling_height,flats_count,floors_total,has_elevator
0,48619,1,6,0.0,0.0,2,False,False,100.000000,58000000.0,1901,1,55.739964,37.630196,3.20,19,6,True
1,52471,1,5,15.0,50.0,4,False,False,133.000000,65273640.0,1901,1,55.739964,37.630196,3.20,19,6,True
2,69229,1,2,0.0,0.0,5,False,False,194.000000,95000000.0,1901,1,55.739964,37.630196,3.20,19,6,True
3,46045,2,2,9.1,80.0,5,False,False,121.900002,45000000.0,1901,1,55.749008,37.585899,3.65,8,4,False
4,54009,3,3,0.0,0.0,4,False,False,120.000000,60000000.0,1901,1,55.759594,37.646488,3.20,26,4,False


**Вывод.** Из базы данных загружена таблица с **141 362 квартирами и 18 столбцами**. В дальнейшем будем предсказывать стоимость квартиры (`price`). Исходный датафрейм `df` оставим без изменений, чтобы при необходимости можно было повторить очистку.

## 2. Анализ и удаление дубликатов

### 2.1. Проверка повторов

In [3]:
# Проверка дубликатов

print("Количество строк:", len(df))

# Полностью одинаковые строки
print("Полные дубликаты:", df.duplicated().sum())

# Повторяющиеся идентификаторы квартир
print(
    "Дубликаты flat_id:",
    df.duplicated(subset=["flat_id"]).sum()
)

# Совпадающие характеристики без учёта ID квартиры
print(
    "Дубликаты без flat_id:",
    df.drop(columns=["flat_id"]).duplicated().sum()
)

Количество строк: 141362
Полные дубликаты: 0
Дубликаты flat_id: 0
Дубликаты без flat_id: 9116


**Вывод.** Полностью одинаковых строк и повторяющихся `flat_id` не обнаружено. Но если не учитывать `flat_id`, получаем **9 116 повторов**. Это не обязательно одни и те же квартиры, однако в этой работе будем удалять такие совпадения.

### 2.2. Характер повторяющихся объявлений

In [4]:
# Группируем квартиры по всем признакам, кроме flat_id

columns = df.columns.drop("flat_id").tolist()

duplicates = (
    df.groupby(columns, dropna=False)
    .size()
    .reset_index(name="count")
)

# Оставляем группы, в которых больше одной квартиры
duplicates = duplicates[duplicates["count"] > 1]

# Сортируем по количеству совпадений
duplicates = duplicates.sort_values(
    by="count",
    ascending=False
)

print("Групп совпадающих квартир:", len(duplicates))
print("Максимальное количество совпадений:", duplicates["count"].max())

display(
    duplicates[
        [
            "building_id",
            "floor",
            "rooms",
            "total_area",
            "kitchen_area",
            "living_area",
            "price",
            "count"
        ]
    ].head(10)
)

Групп совпадающих квартир: 8309
Максимальное количество совпадений: 10


,building_id,floor,rooms,total_area,kitchen_area,living_area,price,count
12709,3729,6,2,45.000000,10.000000,30.000000,30000000.0,10
121698,23510,19,2,56.700001,10.000000,22.000000,22000000.0,8
114534,22627,9,2,63.000000,3.000000,0.000000,10950000.0,8
63872,13890,15,2,49.599998,10.700000,28.400000,12800000.0,8
112012,22328,12,1,49.000000,12.000000,0.000000,7500000.0,7
126165,23996,3,3,119.500000,33.599998,41.400002,21629500.0,7
126181,23996,16,3,119.500000,33.599998,41.400002,23183000.0,7
126163,23996,2,3,119.500000,33.599998,41.400002,21988000.0,6
44431,10485,9,3,60.000000,6.000000,38.000000,12600000.0,6
109601,22045,6,7,320.000000,25.000000,220.000000,170000000.0,6


**Вывод.** Найдено **8 309 групп** квартир с одинаковыми характеристиками. В самой большой группе — **10 записей**. Так как номеров квартир в данных нет, нельзя точно определить, относятся ли записи к одному объекту. Для учебной задачи будем считать такие совпадения дубликатами.

### 2.3. Функция `remove_duplicates()`

In [5]:
def remove_duplicates(df):
    df = df.copy()

    # Все столбцы, кроме идентификатора квартиры
    columns = df.columns.drop("flat_id").tolist()

    # Сортируем по ID, чтобы всегда оставлять
    # квартиру с минимальным flat_id
    df = df.sort_values("flat_id")

    # Удаляем повторяющиеся записи
    df = df.drop_duplicates(
        subset=columns,
        keep="first"
    )

    return df.reset_index(drop=True)


# Применяем функцию
df_clean = remove_duplicates(df)

print("До очистки:", len(df))
print("После очистки:", len(df_clean))
print("Удалено:", len(df) - len(df_clean))

До очистки: 141362
После очистки: 132246
Удалено: 9116


**Вывод.** Функция удаляет совпадающие записи и оставляет в каждой группе квартиру с наименьшим `flat_id`. В результате удалено **9 116 строк**, осталось **132 246 квартир**. При повторном запуске результат будет таким же.

### 2.4. Контроль дубликатов после очистки

In [6]:
columns = df_clean.columns.drop("flat_id").tolist()

print(
    "Оставшиеся дубликаты:",
    df_clean.duplicated(subset=columns).sum()
)

print(
    "Дубликаты flat_id:",
    df_clean.duplicated(subset=["flat_id"]).sum()
)

Оставшиеся дубликаты: 0
Дубликаты flat_id: 0


**Вывод.** После очистки проверим данные ещё раз. Повторов по выбранным характеристикам больше нет, а все значения `flat_id` остаются уникальными.

## 3. Пропуски и логические ошибки площадей

### 3.1. Явные и скрытые пропуски

In [7]:
checks = {
    "Всего квартир": len(df_clean),

    "Пропуски NaN":
        df_clean.isna().sum().sum(),

    "Нулевая площадь кухни":
        (df_clean["kitchen_area"] == 0).sum(),

    "Нулевая жилая площадь":
        (df_clean["living_area"] == 0).sum(),

    "Обе площади равны нулю":
        (
            (df_clean["kitchen_area"] == 0) &
            (df_clean["living_area"] == 0)
        ).sum(),

    "Нулевая кухня не у студий":
        (
            (df_clean["kitchen_area"] == 0) &
            (~df_clean["studio"])
        ).sum(),

    "Нулевая жилая площадь не у студий":
        (
            (df_clean["living_area"] == 0) &
            (~df_clean["studio"])
        ).sum(),

    "Жилая площадь больше общей":
        (
            df_clean["living_area"] >
            df_clean["total_area"]
        ).sum(),

    "Кухня больше общей площади":
        (
            df_clean["kitchen_area"] >
            df_clean["total_area"]
        ).sum(),

    "Сумма площадей больше общей":
        (
            df_clean["living_area"] +
            df_clean["kitchen_area"] >
            df_clean["total_area"] + 0.01
        ).sum(),

    "Этаж выше этажности дома":
        (
            df_clean["floor"] >
            df_clean["floors_total"]
        ).sum()
}

results = pd.DataFrame(
    checks.items(),
    columns=["Проверка", "Количество"]
)

results["Доля, %"] = (
    results["Количество"] / len(df_clean) * 100
).round(2)

display(results)

,Проверка,Количество,"Доля, %"
0,Всего квартир,132246,100.00
1,Пропуски NaN,0,0.00
2,Нулевая площадь кухни,10965,8.29
3,Нулевая жилая площадь,17069,12.91
4,Обе площади равны нулю,7900,5.97
5,Нулевая кухня не у студий,10965,8.29
6,Нулевая жилая площадь не у студий,17069,12.91
7,Жилая площадь больше общей,0,0.00
8,Кухня больше общей площади,2,0.00
9,Сумма площадей больше общей,123,0.09


**Вывод.** Обычных пропусков (`NaN`) в данных нет, но у многих квартир площадь кухни или жилая площадь указана как **0**. Для кухни таких значений **10 965**, для жилой площади — **17 069**, причём у **7 900** квартир обе площади нулевые. Также найдено **123** случая, когда сумма площадей превышает общую, и **2** случая, когда кухня больше всей квартиры. Ошибок с этажностью нет.

### 3.2. Изучение конфликтующих площадей

In [8]:
# Проверяем количество студий
print("Количество студий:")
print(df_clean["studio"].value_counts(dropna=False))

# Квартиры с кухней больше общей площади
bad_kitchen = df_clean[
    df_clean["kitchen_area"] > df_clean["total_area"]
]

print("\nКвартиры с кухней больше общей площади:")
display(
    bad_kitchen[
        ["flat_id", "rooms", "total_area",
         "kitchen_area", "living_area", "price"]
    ]
)

# Проверяем расхождение между площадями
df_check = df_clean.copy()

df_check["area_difference"] = (
    df_check["living_area"]
    + df_check["kitchen_area"]
    - df_check["total_area"]
)

bad_sum = df_check[
    df_check["area_difference"] > 0.01
]

print("\nКоличество ошибок суммы площадей:", len(bad_sum))

print("\nРаспределение величины расхождений:")
display(bad_sum["area_difference"].describe())

print("\n10 квартир с наибольшим расхождением:")
display(
    bad_sum.sort_values(
        "area_difference", ascending=False
    )[
        ["flat_id", "rooms", "total_area",
         "kitchen_area", "living_area",
         "area_difference"]
    ].head(10)
)

Количество студий:
studio
False    132246
Name: count, dtype: int64

Квартиры с кухней больше общей площади:


,flat_id,rooms,total_area,kitchen_area,living_area,price
107757,114197,3,59.0,65.0,43.0,7800000.0
108013,114453,1,39.0,102.0,20.0,5900000.0



Количество ошибок суммы площадей: 123

Распределение величины расхождений:


count    123.000000
mean       7.403089
std        8.908812
min        0.500000
25%        4.000001
50%        6.000000
75%        8.400001
max       83.000000
Name: area_difference, dtype: float64


10 квартир с наибольшим расхождением:


,flat_id,rooms,total_area,kitchen_area,living_area,area_difference
108013,114453,1,39.000000,102.00,20.0,83.000000
107757,114197,3,59.000000,65.00,43.0,49.000000
53600,55600,3,162.399994,45.00,145.0,27.600006
28993,29747,3,73.000000,32.00,62.0,21.000000
67027,69828,1,51.299999,20.00,49.0,17.700001
13882,14055,6,154.600000,18.80,152.3,16.500000
129823,138545,3,103.800000,40.00,80.0,16.200000
96928,102289,2,46.000000,24.00,38.0,16.000000
568,572,3,72.900002,21.00,65.0,13.099998
129154,137802,1,51.000000,16.25,47.5,12.750000


**Вывод.** Во всех **132 246** записях поле `studio` имеет значение `False`. Среди **123** случаев, когда сумма площадей превышает общую, медианное превышение составляет **6 м²**, а максимальное — **83 м²**. Такие расхождения трудно объяснить округлением, поэтому эти записи удалим. Особенно заметны ошибки у квартир с `flat_id` 114197 и 114453.

### 3.3. Квартиры с неустранимым конфликтом площадей

In [9]:
# Диагностика ещё до заполнения нулей медианами.
# Случаи, в которых жилая площадь отсутствует, а кухня занимает всю квартиру.
remaining = df_clean.loc[
    (df_clean["living_area"] == 0)
    & (df_clean["kitchen_area"] >= df_clean["total_area"])
    & ((df_clean["living_area"] + df_clean["kitchen_area"])
       <= df_clean["total_area"] + 0.01)
].copy()

remaining["free_area"] = remaining["total_area"] - remaining["kitchen_area"]

print("Квартир с неустранимым конфликтом:", len(remaining))
display(remaining[[
    "flat_id", "rooms", "total_area", "kitchen_area", "living_area", "free_area"
]].head(15))

Квартир с неустранимым конфликтом: 12


,flat_id,rooms,total_area,kitchen_area,living_area,free_area
19354,19702,1,28.000000,28.000000,0.0,0.0
34439,35426,1,15.000000,15.000000,0.0,0.0
34884,35890,1,14.000000,14.000000,0.0,0.0
34906,35913,2,23.000000,23.000000,0.0,0.0
38663,39800,1,26.000000,26.000000,0.0,0.0
38932,40074,1,37.000000,37.000000,0.0,0.0
38956,40098,1,16.000000,16.000000,0.0,0.0
39024,40169,1,38.000000,38.000000,0.0,0.0
41547,42905,2,31.799999,31.799999,0.0,0.0
41571,42929,1,33.000000,33.000000,0.0,0.0


**Вывод.** Также найдены **12 квартир**, у которых жилая площадь равна нулю, а кухня занимает всю квартиру. Простая подстановка медианного значения здесь не поможет: общая площадь уже полностью занята кухней. Поэтому удалим эти записи вместе со **123** другими квартирами с ошибками площадей. Всего — **135 квартир**.

### 3.4. Функция `remove_invalid_areas()`

In [10]:
def remove_invalid_areas(df):
    df = df.copy()

    area_sum = df["living_area"] + df["kitchen_area"]

    # Логически противоречивые площади
    invalid_areas = (
        (df["living_area"] > df["total_area"])
        | (df["kitchen_area"] > df["total_area"])
        | (area_sum > df["total_area"] + 0.01)
    )

    # Нулевая жилая площадь при кухне,
    # занимающей всю квартиру
    impossible_imputation = (
        (df["living_area"] == 0)
        & (df["kitchen_area"] >= df["total_area"])
    )

    invalid_mask = invalid_areas | impossible_imputation

    print("Удалено некорректных записей:", invalid_mask.sum())

    return df.loc[~invalid_mask].reset_index(drop=True)

**Вывод.** В функции `remove_invalid_areas()` собраны основные проверки площадей. Она удаляет **123 записи** с неправильным соотношением площадей и ещё **12 записей**, где нельзя корректно заполнить жилую площадь, не изменяя уже известные значения.

### 3.5. Удаление некорректных площадей

In [11]:
df_clean = remove_invalid_areas(df_clean)
print("После удаления ошибок площадей:", df_clean.shape)

Удалено некорректных записей: 135
После удаления ошибок площадей: (132111, 18)


**Вывод.** После удаления **135 квартир** с ошибками площадей осталось **132 111 записей**. Повторяющиеся записи к этому моменту уже удалены.

### 3.6. Функция `clean_missing_values()`

In [12]:
def clean_missing_values(df):
    df = df.copy()

    columns = ["kitchen_area", "living_area"]

    for column in columns:
        # Запоминаем, где были нулевые значения
        df[f"{column}_missing"] = (df[column] == 0)

        # Заменяем нули на NaN
        df[column] = df[column].replace(0, np.nan)

        # Медиана по количеству комнат
        median_by_rooms = df.groupby("rooms")[column].transform("median")

        # Заполняем пропуски
        df[column] = df[column].fillna(median_by_rooms)

        # Если в группе нет медианы, используем общую
        df[column] = df[column].fillna(df[column].median())

    # Проверяем сумму площадей после заполнения
    area_sum = df["kitchen_area"] + df["living_area"]
    invalid = area_sum > df["total_area"]

    kitchen_missing = df["kitchen_area_missing"]
    living_missing = df["living_area_missing"]

    # Если отсутствовала только площадь кухни
    mask = invalid & kitchen_missing & ~living_missing

    df.loc[mask, "kitchen_area"] = (
        df.loc[mask, "total_area"]
        - df.loc[mask, "living_area"]
    ) * 0.95

    # Если отсутствовала только жилая площадь
    mask = invalid & living_missing & ~kitchen_missing

    df.loc[mask, "living_area"] = (
        df.loc[mask, "total_area"]
        - df.loc[mask, "kitchen_area"]
    ) * 0.95

    # Если отсутствовали обе площади
    mask = invalid & kitchen_missing & living_missing

    scale = (
        df.loc[mask, "total_area"] * 0.95
        / area_sum.loc[mask]
    )

    df.loc[mask, "kitchen_area"] *= scale
    df.loc[mask, "living_area"] *= scale

    return df

**Вывод.** Для заполнения нулевых площадей создана отдельная функция. Сначала она отмечает исходные нулевые значения в двух новых столбцах, затем подставляет медианную площадь для квартир с таким же количеством комнат. Если такой медианы нет, используется общая медиана. Кроме того, проверяется, чтобы сумма жилой площади и кухни не превышала общую площадь. Коэффициент **0,95** используется как техническое ограничение, а не как точная оценка площади.

### 3.7. Заполнение пропусков в площадях

In [13]:
df_clean = clean_missing_values(df_clean)
print("Размер после заполнения:", df_clean.shape)
print("Осталось NaN:", int(df_clean.isna().sum().sum()))
print("Нулевых кухонь:", int((df_clean["kitchen_area"] == 0).sum()))
print("Нулевых жилых площадей:", int((df_clean["living_area"] == 0).sum()))

Размер после заполнения: (132111, 20)
Осталось NaN: 0
Нулевых кухонь: 0
Нулевых жилых площадей: 0


**Вывод.** После заполнения площадей осталось **132 111 квартир**, а количество столбцов увеличилось с **18 до 20** за счёт двух признаков исходных пропусков. Проверка показала, что нулевых площадей и пропусков больше нет.

### 3.8. Контроль корректности площадей после заполнения

In [14]:
print("Количество квартир:", len(df_clean))

print("Пропуски:", df_clean.isna().sum().sum())

print(
    "Нулевые кухни:",
    (df_clean["kitchen_area"] == 0).sum()
)

print(
    "Нулевые жилые площади:",
    (df_clean["living_area"] == 0).sum()
)

print(
    "Ошибки суммы площадей:",
    (
        df_clean["kitchen_area"]
        + df_clean["living_area"]
        > df_clean["total_area"] + 0.01
    ).sum()
)

print(
    "Дубликаты:",
    df_clean.drop(columns=["flat_id"]).duplicated().sum()
)

Количество квартир: 132111
Пропуски: 0
Нулевые кухни: 0
Нулевые жилые площади: 0
Ошибки суммы площадей: 0
Дубликаты: 0


**Вывод.** Ещё раз проверим результат заполнения. Пропусков, нулевых площадей, ошибок суммы площадей и дубликатов не осталось. Теперь можно переходить к анализу цен. При этом важно учитывать, что часть площадей заполнена приблизительно, а не взята из исходных объявлений.

## 4. Анализ и удаление выбросов стоимости

### 4.1. Распределение цены квартиры и цены квадратного метра

In [15]:
# Создаём копию очищенных данных для анализа выбросов
df_outliers = df_clean.copy()

# Стоимость одного квадратного метра
df_outliers["price_per_sqm"] = (
    df_outliers["price"] / df_outliers["total_area"]
)

# Рассчитываем процентили
percentiles = [0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]

stats = df_outliers[
    ["price", "price_per_sqm"]
].quantile(percentiles)

display(stats.round(0))

,price,price_per_sqm
0.01,5150000.0,115625.0
0.05,6500000.0,145819.0
0.25,8900000.0,191327.0
0.50,11800000.0,233333.0
0.75,16850000.0,287879.0
0.95,48000000.0,469665.0
0.99,151678125.0,875000.0


**Вывод.** Медианная стоимость квартиры составляет **11,8 млн ₽**, а медианная цена квадратного метра — примерно **233 333 ₽**. У 99% квартир цена за метр не превышает **875 000 ₽**. Заметим, что распределение цен включает и значительно более дорогие квартиры, поэтому отдельно посмотрим на крайние значения.

### 4.2. Примеры наиболее подозрительных цен

In [16]:
columns = [
    "flat_id",
    "rooms",
    "total_area",
    "price",
    "price_per_sqm"
]

print("10 квартир с минимальной ценой за м²:")

display(
    df_outliers.nsmallest(
        10, "price_per_sqm"
    )[columns]
)

print("10 квартир с максимальной ценой за м²:")

display(
    df_outliers.nlargest(
        10, "price_per_sqm"
    )[columns]
)

10 квартир с минимальной ценой за м²:


,flat_id,rooms,total_area,price,price_per_sqm
66771,69633,2,42.0,11.0,0.261905
101276,107198,2,70.0,19.0,0.271429
98916,104573,1,59.0,85.0,1.440678
58125,60417,1,36.0,1500.0,41.666667
127381,135997,2,95.0,4000.0,42.105263
21219,21821,2,60.0,3000.0,50.000000
120988,128961,1,40.0,2000.0,50.000000
73620,76800,1,38.0,2000.0,52.631579
122148,130229,2,54.0,3000.0,55.555556
86162,90369,2,70.0,3890.0,55.571429


10 квартир с максимальной ценой за м²:


,flat_id,rooms,total_area,price,price_per_sqm
80034,83773,2,37.0,9.799999e+09,2.648649e+08
39639,40855,2,45.0,9.200000e+09,2.044444e+08
48705,50455,3,103.0,4.686425e+09,4.549927e+07
81918,85787,5,220.0,9.873738e+09,4.488063e+07
111231,118083,4,208.0,8.147034e+09,3.916843e+07
75877,79266,4,120.0,4.447152e+09,3.705960e+07
75876,79264,3,97.0,3.525905e+09,3.634953e+07
36435,37521,3,84.0,2.500000e+09,2.976190e+07
56547,58778,7,202.0,4.048056e+09,2.003988e+07
53896,55967,3,61.0,1.174683e+09,1.925710e+07


**Вывод.** Среди квартир встречаются подозрительные цены: некоторые объекты стоят всего **11–19 ₽**, а для других цена квадратного метра достигает сотен миллионов рублей. Возможно, часть таких значений появилась из-за ошибок в данных. Перед обучением модели их стоит проверить, чтобы они не влияли на результат.

### 4.3. Анализ количества наблюдений по порогам

In [17]:
# Анализ экстремальных значений стоимости квадратного метра

price_sqm = df_outliers["price_per_sqm"]

thresholds = {
    "Меньше 20 тыс. руб./м²": (price_sqm < 20_000).sum(),
    "Меньше 50 тыс. руб./м²": (price_sqm < 50_000).sum(),
    "Больше 2 млн руб./м²": (price_sqm > 2_000_000).sum(),
    "Больше 3 млн руб./м²": (price_sqm > 3_000_000).sum(),
    "Больше 5 млн руб./м²": (price_sqm > 5_000_000).sum(),
    "Больше 10 млн руб./м²": (price_sqm > 10_000_000).sum()
}

threshold_results = pd.DataFrame(
    thresholds.items(),
    columns=["Диапазон", "Количество квартир"]
)

display(threshold_results)

,Диапазон,Количество квартир
0,Меньше 20 тыс. руб./м²,153
1,Меньше 50 тыс. руб./м²,194
2,Больше 2 млн руб./м²,94
3,Больше 3 млн руб./м²,29
4,Больше 5 млн руб./м²,18
5,Больше 10 млн руб./м²,15


**Вывод.** Посмотрим, сколько квартир имеют очень низкую или высокую цену за квадратный метр. Дешевле **20 000 ₽/м²** оказалось **153 квартиры**, дороже **5 млн ₽/м²** — **18**. При этом **94 квартиры** стоят дороже **2 млн ₽/м²**, но не все такие значения обязательно ошибочны: среди них могут быть действительно дорогие объекты.

### 4.4. Сравнение с методом IQR

In [18]:
Q1 = price_sqm.quantile(0.25)
Q3 = price_sqm.quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f"Нижняя граница IQR: {lower_bound:,.0f} руб./м²")
print(f"Верхняя граница IQR: {upper_bound:,.0f} руб./м²")

print(
    "Количество выбросов по IQR:",
    ((price_sqm < lower_bound) |
     (price_sqm > upper_bound)).sum()
)

Нижняя граница IQR: 46,500 руб./м²
Верхняя граница IQR: 432,706 руб./м²
Количество выбросов по IQR: 8491


**Вывод.** Метод IQR даёт границы примерно от **46 500 до 432 706 ₽/м²**. За пределы этого диапазона выходят **8 491 квартира** (около **6,4%**). Заметим, что удаление всех таких квартир может привести к потере реальных дорогих объектов. Поэтому для этой работы используем более мягкие границы.

### 4.5. Функция `remove_outliers()`

In [19]:
def remove_outliers(df):
    df = df.copy()

    # Стоимость квадратного метра
    price_per_sqm = df["price"] / df["total_area"]

    # Границы допустимой стоимости
    lower_bound = 20_000
    upper_bound = 5_000_000

    # Определяем выбросы
    outlier_mask = (
        (price_per_sqm < lower_bound)
        | (price_per_sqm > upper_bound)
    )

    print("Найдено выбросов:", outlier_mask.sum())

    # Удаляем выбросы
    df = df.loc[~outlier_mask].copy()

    return df.reset_index(drop=True)

**Вывод.** Функция удаляет квартиры с ценой за квадратный метр ниже **20 000 ₽/м²** или выше **5 млн ₽/м²**. Эти пороги подходят для выбранной учебной задачи, но не являются универсальными. Цену за квадратный метр не будем добавлять в признаки для модели, так как она вычисляется через `price` — величину, которую нужно предсказать.

## 5. Единый воспроизводимый алгоритм

### 5.1. Функция `clean_data()`

In [20]:
def clean_data(df):
    df = remove_duplicates(df)
    df = remove_invalid_areas(df)
    df = clean_missing_values(df)
    df = remove_outliers(df)

    return df

**Вывод.** Все этапы объединены в функцию `clean_data()`: сначала удаляются дубликаты, затем записи с ошибками площадей, после этого заполняются нулевые площади и в конце убираются ценовые выбросы. Такой же порядок нужно сохранить в `src/cleaning.py` и при запуске очистки через Airflow.

### 5.2. Полная очистка из исходного DataFrame

In [21]:
# Полная очистка исходного датасета
df_final = clean_data(df)

print("Исходный размер:", df.shape)
print("Итоговый размер:", df_final.shape)

print("Осталось пропусков:", df_final.isna().sum().sum())

print(
    "Дубликатов:",
    df_final.drop(columns=["flat_id"]).duplicated().sum()
)

print(
    "Ошибок площадей:",
    (
        df_final["kitchen_area"] + df_final["living_area"]
        > df_final["total_area"] + 0.01
    ).sum()
)

Удалено некорректных записей: 135
Найдено выбросов: 171
Исходный размер: (141362, 18)
Итоговый размер: (131940, 20)
Осталось пропусков: 0
Дубликатов: 0
Ошибок площадей: 0


**Вывод.** Запустим полную очистку ещё раз, начиная с исходного датафрейма. Удалено **9 116 дубликатов**, **135 записей** с ошибками площадей и **171** ценовой выброс. В результате получилось **131 940 строк и 20 столбцов**. Исходная таблица при этом осталась без изменений.

### 5.3. Финальная проверка качества

In [22]:
# Проверяем единообразие результата перед загрузкой в PostgreSQL.
price_sqm_final = df_final["price"] / df_final["total_area"]

checks_final = {
    "Пропуски": int(df_final.isna().sum().sum()),
    "Дубликаты по характеристикам": int(
        df_final.drop(columns=["flat_id"]).duplicated().sum()
    ),
    "Дубликаты flat_id": int(df_final["flat_id"].duplicated().sum()),
    "Нулевые и отрицательные площади": int(
        ((df_final["kitchen_area"] <= 0) | (df_final["living_area"] <= 0)).sum()
    ),
    "Ошибки суммы площадей": int(
        ((df_final["kitchen_area"] + df_final["living_area"])
         > df_final["total_area"] + 0.01).sum()
    ),
    "Выбросы за принятыми границами": int(
        ((price_sqm_final < 20_000) | (price_sqm_final > 5_000_000)).sum()
    ),
}

display(pd.DataFrame(checks_final.items(), columns=["Проверка", "Количество"]))
assert df_final.shape == (131940, 20)
assert all(value == 0 for value in checks_final.values())

,Проверка,Количество
0,Пропуски,0
1,Дубликаты по характеристикам,0
2,Дубликаты flat_id,0
3,Нулевые и отрицательные площади,0
4,Ошибки суммы площадей,0
5,Выбросы за принятыми границами,0


**Вывод.** Итоговая проверка прошла успешно: пропусков, дубликатов, неправильных площадей и цен за выбранными границами не осталось. Размер данных совпал с ожидаемым. Проверки `assert` помогут заметить ошибки, если в дальнейшем изменятся данные или код.

## 6. Итоги очистки данных

| Этап | Удалено | Осталось |
|:--|--:|--:|
| Исходный датасет | — | 141 362 |
| Удаление совпадений по характеристикам | 9 116 | 132 246 |
| Логические ошибки площадей | 135 | 132 111 |
| Экстремальные значения стоимости за м² | 171 | **131 940** |

**Общий вывод.** В этой работе проверены данные о квартирах и подготовлены функции для их очистки. Сначала удалены повторяющиеся записи, затем — квартиры с ошибками площадей. Нулевые значения площадей заполнены, а слишком низкие и высокие цены отфильтрованы. Из **141 362 квартир** осталось **131 940**, то есть удалено **9 422 записи (около 6,67%)**. Также добавлены два столбца, которые показывают, где площадь кухни или жилая площадь изначально была нулевой.

**Что важно учитывать дальше:**

- Квартиры с одинаковыми характеристиками не обязательно являются дубликатами: номеров квартир в таблице нет.
- Нулевые площади заполнены приблизительно, поэтому полученные значения могут отличаться от реальных.
- Границы **20 тыс. – 5 млн ₽/м²** выбраны для этой работы. Для другой выборки их, возможно, придётся изменить.
- При обучении модели значения для заполнения пропусков нужно рассчитывать только по обучающей выборке, чтобы не использовать информацию из тестовой.